In [1]:
from ultralytics import YOLO
import easyocr
import cv2
import csv
import os

In [2]:
def processing_plate(image):
    img_rsz = cv2.resize(image, (344,112))
    img_gray = cv2.cvtColor(img_rsz, cv2.COLOR_BGR2GRAY)
    img_denoised = cv2.GaussianBlur(img_gray, (5, 5), 0)
    img_edge = cv2.Canny(img_denoised,100,200)
    # img_final = cv2.cvtColor(img_edge, cv2.COLOR_GRAY2BGR)

    return img_edge

def read_license_plate(reader, image):
    detections = reader.readtext(image)
    for detection in detections:
        bbox, text, score = detection
        text = text.upper().replace(' ', '')
        return text, score
    return None, None

def border_line(frame, start_point, end_point, color=(0, 255, 0), thickness=4):
    frame = cv2.line(frame, start_point, end_point, color=color, thickness=thickness)
    return frame

def write_csv(file_path, track_id, vehicle_bbox, vehicle_bbox_score, plate_bbox, plate_bbox_score, text_plate, text_plate_score):
    """Write vehicle and plate information to a CSV file."""
    # Ensure the directory exists
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    # Append data to the CSV file
    with open(file_path, mode='a', newline='') as csv_file:
        csv_writer = csv.writer(csv_file)
        
        # Write header if the file is empty
        if csv_file.tell() == 0:
            csv_writer.writerow(['track_id', 'vehicle_bbox', 'vehicle_bbox_score', 'plate_bbox', 'plate_bbox_score', 'text_plate', 'text_plate_score'])
        
        # Write the data
        csv_writer.writerow([track_id, vehicle_bbox, vehicle_bbox_score, plate_bbox, plate_bbox_score, text_plate, text_plate_score])


In [9]:
video_path = "video/traffic.mp4"
video = cv2.VideoCapture(video_path)

start_point = (0, int(video.get(4))-300)
end_point = (int(video.get(3)), int(video.get(4))-300)

coco_model = YOLO('model/yolov8n.pt')
np_model = YOLO('model/license_plate_detector.pt', task='detect')
reader = easyocr.Reader(['en'], gpu=False)

Using CPU. Note: This module is much faster with a GPU.


In [10]:
ret = True
saved_ids = set()
plate_text_dict = {}

while ret:
    ret, frame = video.read()
    frame = border_line(frame, start_point, end_point)

    if ret:
        # Deteksi kendaraan
        detections = coco_model.track(frame, persist=True, conf=0.5, classes=[2])[0]
        for detection in detections.boxes.data.tolist():
            x1, y1, x2, y2, track_id, score, class_id = detection

            if y2 > start_point[1]:
                vehicle_bounding_boxes = []
                vehicle_bounding_boxes.append([x1, y1, x2, y2, track_id, score])

                # Ambil gambar kendaraan
                for bbox in vehicle_bounding_boxes:
                    roi = frame[int(y1):int(y2), int(x1):int(x2)]
                    
                    # Deteksi plat
                    license_plates = np_model(roi)[0]  # Detektor plat dari roi (image kendaraan)

                    for license_plate in license_plates.boxes.data.tolist():
                        plate_x1, plate_y1, plate_x2, plate_y2, plate_score, _ = license_plate
                        plate = roi[int(plate_y1):int(plate_y2), int(plate_x1):int(plate_x2)]
                        plate_gray = cv2.cvtColor(plate, cv2.COLOR_BGR2GRAY)
                        np_text, np_score = read_license_plate(reader=reader, image=plate_gray)
                        
                        if track_id not in saved_ids: # not to save the same image
                            vehicle_bbox = f"{x1},{y1},{x2},{y2}"
                            vehicle_bbox_score = score

                            plate_bbox = f"{plate_x1},{plate_y1},{plate_x2},{plate_y2}"
                            plate_bbox_score = plate_score

                            text_plate = np_text
                            text_plate_score = np_score
                            
                            write_csv('result/vehicle_data.csv', track_id, 
                                      vehicle_bbox, vehicle_bbox_score, 
                                      plate_bbox, plate_bbox_score,
                                      text_plate, text_plate_score)
                            
                            cv2.imwrite('result/' + str(track_id) + '_car.jpg', roi)
                            cv2.imwrite('result/' + str(track_id) + '_plate.jpg', plate)

                            saved_ids.add(track_id)
                            plate_text_dict[track_id] = text_plate

                        # Gambar bounding box plat nomor
                        # cv2.rectangle(frame, (int(x1 + plate_x1), int(y1 + plate_y1)), (int(x1 + plate_x2), int(y1 + plate_y2)), (0, 255, 0), 2)
                        # cv2.putText(frame, f'Text Plate: {text_plate}', (int(x1 + plate_x1), int(y1 + plate_y1 - 10)), 
                        #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

                # Gambar bounding box kendaraan
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
                cv2.putText(frame, f'Car ID: {track_id}', (int(x1), int(y1 - 30)), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                
                if track_id in plate_text_dict: # ambil text_plate dari dictionary plate_text_dict
                    cv2.putText(frame, f'Text Plate: {plate_text_dict[track_id]}', (int(x1), int(y1 - 10)),  
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
                else: # jika text_plate belum ada di dictionary, gunakan nilai terbaru
                    cv2.putText(frame, f'Text Plate: {text_plate}', (int(x1), int(y1 - 10)),  
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2) 

        # Menampilkan frame video dengan deteksi
        cv2.imshow('Video', frame)

    # Menunggu input kunci untuk keluar
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

video.release()
cv2.destroyAllWindows()


0: 384x640 2 cars, 132.2ms
Speed: 0.0ms preprocess, 132.2ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 576x640 1 license_plate, 150.1ms
Speed: 0.0ms preprocess, 150.1ms inference, 0.0ms postprocess per image at shape (1, 3, 576, 640)

0: 384x640 2 cars, 163.6ms
Speed: 10.4ms preprocess, 163.6ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 576x640 1 license_plate, 177.7ms
Speed: 18.4ms preprocess, 177.7ms inference, 0.0ms postprocess per image at shape (1, 3, 576, 640)

0: 384x640 2 cars, 115.8ms
Speed: 0.0ms preprocess, 115.8ms inference, 0.0ms postprocess per image at shape (1, 3, 384, 640)

0: 576x640 1 license_plate, 149.0ms
Speed: 0.0ms preprocess, 149.0ms inference, 17.5ms postprocess per image at shape (1, 3, 576, 640)

0: 384x640 2 cars, 138.3ms
Speed: 0.0ms preprocess, 138.3ms inference, 11.8ms postprocess per image at shape (1, 3, 384, 640)

0: 576x640 1 license_plate, 147.8ms
Speed: 16.8ms preprocess, 147.8ms inference, 0.0ms

**NOTED** : Perbaiki bug masalah text plate, text_plate akan berubah mengikuti text_plate kendaraan yang baru

In [11]:
import pandas as pd
data = pd.read_csv("result/vehicle_data.csv")
data

,track_id,vehicle_bbox,vehicle_bbox_score,plate_bbox,plate_bbox_score,text_plate,text_plate_score
0,1.0,"248.22412109375,229.493408203125,522.328186035...",0.869316,"75.58023071289062,188.81698608398438,137.16142...",0.705153,44NJZ94B,0.004910
1,3.0,"799.8897094726562,146.87570190429688,1049.9259...",0.546461,"66.58291625976562,233.91534423828125,129.96499...",0.753593,GARGOGL,0.275770
2,4.0,"498.0447082519531,329.9162902832031,868.852172...",0.800632,"76.16244506835938,316.037109375,154.7071685791...",0.783581,GAR6OGL],0.314053
3,5.0,"840.3004150390625,154.5867156982422,1127.99792...",0.856539,"79.38691711425781,210.83834838867188,143.46499...",0.800805,WAE6705,0.421275
4,6.0,"911.9093017578125,204.9300079345703,1176.68151...",0.886961,"89.31842803955078,165.10609436035156,148.47111...",0.715146,J4PHI88,0.194628
5,7.0,"681.3970336914062,201.5623321533203,963.042053...",0.681213,"76.98372650146484,169.5500030517578,143.273025...",0.764920,A3K961,0.510456
6,8.0,"1014.99365234375,192.2427215576172,1273.433593...",0.826735,"83.72577667236328,174.75045776367188,149.49925...",0.757051,7496886,0.522534
7,10.0,"746.26220703125,203.12179565429688,1007.708312...",0.822094,"81.27412414550781,159.55441284179688,147.72740...",0.765454,V6L90,0.173991
